In [3]:
import xarray as xr
import pandas as pd
import datetime as dt
import tqdm
import numpy as np
import matplotlib.pyplot as plt
import pyshtools

# Load predictions
predictions = xr.open_dataset("/dcai/projects01/cu_0003/user_space/has/predictions.zarr", engine="zarr")

# Load ground truth
ground_truth = xr.open_dataset("/dcai/projects01/cu_0003/user_space/has/git-repos/mllam/mllam-exps-ShCu3/data/datastore.interior.zarr", engine="zarr")

In [4]:
def combine_state_features(ds):
    """
    Combine individual state features from an xarray.Dataset into a single dataset.

    Parameters:
    ds (xarray.Dataset): The input dataset containing state features.

    Returns:
    xarray.Dataset: A new dataset with each state feature as a separate variable.
    """
    # Create a new dataset to hold the combined features
    combined_ds = xr.Dataset()

    # Loop through each state feature and add it to the combined dataset
    for feature in ds.state_feature.values:
        individual_array = ds.state.sel(state_feature=feature)
        combined_ds[feature] = individual_array

    return combined_ds

In [5]:
combined_predictions = combine_state_features(predictions)
combined_ground_truth = combine_state_features(ground_truth)

frequency = "1h"  # Frequency of forecasts
length_of_forecast = 15  # Length of forecast in time steps

In [ ]:
for forecast, init_time in enumerate(tqdm.tqdm(pd.date_range(
        start=predictions.start_time.min().values,
        end=predictions.start_time.max().values,
        freq=frequency,
    ))):
    # Extract the forecast time slice
    try:
        ground_truth = combined_ground_truth.sel(time=slice(init_time+dt.timedelta(minutes=10), init_time + dt.timedelta(minutes=10*length_of_forecast)))
        predictions = combined_predictions.sel(start_time=init_time).isel(elapsed_forecast_duration=slice(0, length_of_forecast))
    except KeyError:
        print(f"KeyError for init_time: {init_time}, skipping...")
        continue
    assert len(predictions.time) == length_of_forecast, f"Forecast length mismatch: {len(predictions.time)} != {length_of_forecast}"
    assert len(ground_truth.time) == length_of_forecast, f"Ground truth length mismatch: {len(ground_truth.time)} != {length_of_forecast}"

    preds = predictions.rename({'elapsed_forecast_duration': 'leadtime',}).expand_dims('forecast').drop_vars(['time', 'start_time', 'state_feature'])
    reference = ground_truth.rename({'time': 'leadtime',}).expand_dims('forecast').drop_vars(['state_feature', 'state_feature_long_name', 'state_feature_units', 'state_feature_source_dataset'])
    reference = reference.assign_coords({'leadtime': pd.date_range(start=init_time+dt.timedelta(minutes=10), periods=length_of_forecast, freq='10min') - init_time})

    if forecast == 0:
        p = preds
        r = reference
    else:
        p = xr.concat([p, preds], dim='forecast')
        r = xr.concat([r, reference], dim='forecast')
    

  0%|          | 1/261 [00:02<12:03,  2.78s/it]

KeyError for init_time: 2020-02-01 01:20:00, skipping...
KeyError for init_time: 2020-02-01 02:20:00, skipping...
KeyError for init_time: 2020-02-01 03:20:00, skipping...


  2%|▏         | 6/261 [00:08<05:58,  1.40s/it]

KeyError for init_time: 2020-02-01 06:20:00, skipping...
KeyError for init_time: 2020-02-01 07:20:00, skipping...


  3%|▎         | 9/261 [00:11<05:20,  1.27s/it]